In [1]:
import os
import sys
import gc
import pickle
import warnings
from pathlib import Path
from typing import Dict, Optional, List, Union

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import torch
from datasets import load_from_disk
from transformers import BertForMaskedLM
from tqdm.auto import tqdm

# ============================================================
# USER PATHS
# ============================================================
GENEFORMER_ROOT = "/ibex/user/chenj0i/Geneformer"
MODEL_PATH = "/ibex/user/chenj0i/Geneformer/Geneformer_pretrained/gf-6L-30M-i2048"
TOKEN_DICT_PATH = "/ibex/user/chenj0i/Geneformer/geneformer/token_dictionary.pkl"

# Root containing pseudo-control strategy folders.
PSEUDO_ROOT = Path(
    "/ibex/project/c2366/Perturb_data/Replogle_k562_data/Replogle_K562_essential/single"
)

# Output folder dedicated to this strict Geneformer-only run.
OUT_ROOT = PSEUDO_ROOT / "_geneformer_embeddings"

TOKENIZED_DIR = OUT_ROOT / "tokenized"
EMB_DIR = OUT_ROOT / "embeddings"
UPDATED_H5AD_DIR = OUT_ROOT / "h5ad_with_embs"
MANIFEST_DIR = OUT_ROOT / "manifests"

TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)
UPDATED_H5AD_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# IMPORT GENEFORMER MODULES
# ------------------------------------------------------------
sys.path.append(GENEFORMER_ROOT)

from geneformer.tokenizer import TranscriptomeTokenizer
from geneformer import perturber_utils as pu

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# RUN CONFIG
# ============================================================

# Pseudo-control h5ad filename used in your strategy folders.
PSEUDO_H5AD_NAME = "pseudo_control_aligned_to_perturbed.h5ad"

# For first debugging, keep one variant and optionally subset cells.
# Use slash-style paths or slug-style names. Use [] to process all discovered variants.
VARIANTS = [
    "S5_SEACell_OT_sampled_average/nmc_500/topk_05/seed_000",
]

# Use None for full data after the debug run works.
SUBSET_N_CELLS = None

# Geneformer input settings.
COUNTS_SOURCE = "X"

# Set this manually if auto-detection picks the wrong column.
# Accepted examples: "ensembl_id", "ensemblid", "gene_id", "gene_symbol", "index".
ENSEMBL_COL = None
ENSEMBL_COL_CANDIDATES = [
    "ensembl_id",
    "ensemblid",
    "gene_id",
    "gene_ids",
    "gene_symbol",
    "feature_id",
    "features",
    "index",
]

# Metadata columns to preserve during tokenization if present.
# For pseudo-control data, perturbation_label is usually the most useful label.
BATCH_KEY = None
LABEL_KEY = "perturbation_label"
EXTRA_OBS_ATTRS = [
    "condition",
    "perturbation",
    "perturbation_label",
    "perturbation_tokens",
    "nperts",
    "gene",
    "guide_id",
    "source_obs_name",
    "adata_order",
]

# Geneformer tokenizer settings, matching the provided script unless changed here.
TOKENIZE_NPROC = 16
TOKENIZE_CHUNK_SIZE = 512
USE_GENERATOR = False

# Embedding extraction settings.
EMB_LAYER = -2
FORWARD_BATCH_SIZE = 64
PAD_TOKEN_ID = 0
OBSM_KEY = "X_geneformer"

# Output behavior.
OVERWRITE_PREPARED = False
OVERWRITE_TOKENIZED = False
OVERWRITE_EMBEDDINGS = False
WRITE_H5AD_WITH_EMB = True
OVERWRITE_H5AD_WITH_EMB = False
CONTINUE_ON_ERROR = True

print(f"[Info] DEVICE = {DEVICE}")
print(f"[Info] PSEUDO_ROOT = {PSEUDO_ROOT}")
print(f"[Info] OUT_ROOT = {OUT_ROOT}")

[Info] DEVICE = cuda
[Info] PSEUDO_ROOT = /ibex/project/c2366/Perturb_data/Replogle_k562_data/Replogle_K562_essential/single
[Info] OUT_ROOT = /ibex/project/c2366/Perturb_data/Replogle_k562_data/Replogle_K562_essential/single/_geneformer_embeddings


In [2]:
# ============================================================
# PATH CHECKS
# ============================================================
paths_to_check = {
    "GENEFORMER_ROOT": Path(GENEFORMER_ROOT),
    "MODEL_PATH": Path(MODEL_PATH),
    "TOKEN_DICT_PATH": Path(TOKEN_DICT_PATH),
    "PSEUDO_ROOT": PSEUDO_ROOT,
    "OUT_ROOT": OUT_ROOT,
}

for name, path in paths_to_check.items():
    print(f"{name}: {path} | exists={path.exists()}")

GENEFORMER_ROOT: /ibex/user/chenj0i/Geneformer | exists=True
MODEL_PATH: /ibex/user/chenj0i/Geneformer/Geneformer_pretrained/gf-6L-30M-i2048 | exists=True
TOKEN_DICT_PATH: /ibex/user/chenj0i/Geneformer/geneformer/token_dictionary.pkl | exists=True
PSEUDO_ROOT: /ibex/project/c2366/Perturb_data/Replogle_k562_data/Replogle_K562_essential/single | exists=True
OUT_ROOT: /ibex/project/c2366/Perturb_data/Replogle_k562_data/Replogle_K562_essential/single/_geneformer_embeddings | exists=True


In [3]:
# ============================================================
# DATASET DISCOVERY FOR PSEUDO-CONTROL FILES
# ============================================================

def variant_to_slug(variant: Union[str, Path]) -> str:
    variant = str(variant).strip().strip("/")
    return variant.replace("/", "__")


def infer_variant_from_path(h5ad_path: Path, root: Path = PSEUDO_ROOT) -> str:
    rel_parent = h5ad_path.parent.relative_to(root)
    return str(rel_parent)


def matches_variant(variant: str, filters: List[str]) -> bool:
    if not filters:
        return True
    slug = variant_to_slug(variant)
    normalized = {str(v).strip().strip("/") for v in filters}
    normalized_slugs = {variant_to_slug(v) for v in normalized}
    return variant in normalized or slug in normalized_slugs


def discover_pseudo_control_h5ads(
    root: Path = PSEUDO_ROOT,
    filename: str = PSEUDO_H5AD_NAME,
    variants: Optional[List[str]] = None,
) -> pd.DataFrame:
    rows = []
    for path in sorted(root.rglob(filename)):
        rel_parts = path.relative_to(root).parts

        # Skip generated output folders.
        if any(part.startswith("_") for part in rel_parts):
            continue

        variant = infer_variant_from_path(path, root=root)
        if not matches_variant(variant, variants or []):
            continue

        rows.append({
            "variant": variant,
            "variant_slug": variant_to_slug(variant),
            "h5ad": str(path),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise FileNotFoundError(
            f"No {filename} files found under {root} for variants={variants}"
        )

    return df.sort_values(["variant"]).reset_index(drop=True)


pseudo_df = discover_pseudo_control_h5ads(PSEUDO_ROOT, PSEUDO_H5AD_NAME, VARIANTS)
print(f"[Discover] Found {len(pseudo_df)} pseudo-control h5ad file(s).")
display(pseudo_df)

[Discover] Found 1 pseudo-control h5ad file(s).


,variant,variant_slug,h5ad
0,S5_SEACell_OT_sampled_average/nmc_500/topk_05/...,S5_SEACell_OT_sampled_average__nmc_500__topk_0...,/ibex/project/c2366/Perturb_data/Replogle_k562...


In [4]:
# ------------------------------------------------------------
# HELPERS FROM PROVIDED SCRIPT, ADAPTED ONLY FOR PSEUDO-CONTROL DATA
# ------------------------------------------------------------

def load_token_dict(token_dict_path: str):
    with open(token_dict_path, "rb") as f:
        return pickle.load(f)


TOKEN_DICT = load_token_dict(TOKEN_DICT_PATH)
VALID_ENSG = set([k for k in TOKEN_DICT.keys() if k not in {"<pad>", "<mask>"}])
print(f"[Info] Loaded Geneformer token dictionary with {len(VALID_ENSG)} valid genes.")


def get_counts_matrix(adata: ad.AnnData, counts_source: str):
    if counts_source == "X":
        X = adata.X
    else:
        if counts_source not in adata.layers:
            raise KeyError(f"Layer '{counts_source}' not found in adata.layers.")
        X = adata.layers[counts_source]
    return X


def compute_n_counts(X):
    if sp.issparse(X):
        return np.asarray(X.sum(axis=1)).ravel()
    return np.asarray(X.sum(axis=1)).ravel()


def clean_ensembl_ids(series: pd.Series) -> pd.Series:
    # remove version suffix like ENSG00000123456.12 -> ENSG00000123456
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.\d+$", "", regex=True)
    )


def get_ensembl_source_series(
    adata: ad.AnnData,
    ensembl_col: Optional[str] = None,
    candidates: Optional[List[str]] = None,
) -> tuple[str, pd.Series]:
    """
    Determine which adata.var field should be used as the Ensembl-ID source.

    This keeps the original script's 'ensembl_col' concept, but lets pseudo-control
    datasets auto-detect the column when the exact field differs across h5ad files.
    """
    candidates = candidates or ENSEMBL_COL_CANDIDATES

    if ensembl_col is not None:
        if ensembl_col == "index":
            return "index", pd.Series(adata.var_names.astype(str), index=adata.var_names)
        if ensembl_col not in adata.var.columns:
            raise KeyError(f"Ensembl source column '{ensembl_col}' not found in adata.var.")
        return ensembl_col, adata.var[ensembl_col]

    for col in candidates:
        if col == "index":
            s = pd.Series(adata.var_names.astype(str), index=adata.var_names)
        elif col in adata.var.columns:
            s = adata.var[col]
        else:
            continue

        cleaned = clean_ensembl_ids(s)
        n_match = int(cleaned.isin(VALID_ENSG).sum())
        if n_match > 0:
            return col, s

    raise ValueError(
        "Could not auto-detect an Ensembl ID column with overlap to Geneformer vocabulary. "
        f"Available adata.var columns: {list(adata.var.columns)}. "
        "Set ENSEMBL_COL manually."
    )


def existing_obs_attrs(adata: ad.AnnData, batch_key: Optional[str], label_key: Optional[str]) -> Dict[str, str]:
    """
    Build custom_attr_name_dict for TranscriptomeTokenizer using only columns
    that actually exist in pseudo-control obs.
    """
    attrs = []

    for key in [batch_key, label_key] + EXTRA_OBS_ATTRS:
        if key is None:
            continue
        if key not in attrs:
            attrs.append(key)

    attr_dict = {}
    for key in attrs:
        if key in adata.obs.columns:
            attr_dict[key] = key

    return attr_dict


def prepare_geneformer_input(
    in_h5ad: Path,
    out_h5ad: Path,
    ensembl_col: Optional[str],
    counts_source: str,
    batch_key: Optional[str],
    label_key: Optional[str],
):
    """
    Create a Geneformer-ready h5ad containing:
      - raw-count-like matrix in adata.X
      - adata.var['ensembl_id']
      - adata.obs['n_counts']
      - retained metadata for batch/label if present
    """
    print(f"\n[Prepare] Reading: {in_h5ad}")
    adata = sc.read_h5ad(in_h5ad)

    if SUBSET_N_CELLS is not None:
        adata = adata[:SUBSET_N_CELLS].copy()
        print(f"[Prepare] Debug subset: first {adata.n_obs} cells")

    # Preserve row order for downstream alignment and plotting.
    if "adata_order" not in adata.obs.columns:
        adata.obs["adata_order"] = np.arange(adata.n_obs, dtype=np.int64)
    if "source_obs_name" not in adata.obs.columns:
        adata.obs["source_obs_name"] = adata.obs_names.astype(str)

    if batch_key is not None and batch_key not in adata.obs.columns:
        print(f"[Prepare][Warn] Batch key '{batch_key}' not found in adata.obs. It will not be tokenized.")

    if label_key is not None and label_key not in adata.obs.columns:
        print(f"[Prepare][Warn] Label key '{label_key}' not found in adata.obs. It will not be tokenized.")

    resolved_ensembl_col, source_series = get_ensembl_source_series(
        adata,
        ensembl_col=ensembl_col,
        candidates=ENSEMBL_COL_CANDIDATES,
    )

    X = get_counts_matrix(adata, counts_source)

    # Create a fresh AnnData with desired X
    obs = adata.obs.copy()
    var = adata.var.copy()

    var["ensembl_id"] = clean_ensembl_ids(source_series).values

    # keep only genes present in Geneformer vocabulary
    keep_gene_mask = var["ensembl_id"].isin(VALID_ENSG).values
    n_before = adata.n_vars
    n_keep = int(keep_gene_mask.sum())

    if n_keep == 0:
        raise ValueError(
            f"No genes matched Geneformer token dictionary for file: {in_h5ad}"
        )

    print(f"[Prepare] Ensembl source column: {resolved_ensembl_col}")
    print(f"[Prepare] Genes before filtering: {n_before}")
    print(f"[Prepare] Genes matched Geneformer vocab: {n_keep}")

    if sp.issparse(X):
        X = X[:, keep_gene_mask].tocsr()
    else:
        X = X[:, keep_gene_mask]

    obs = obs.copy()
    var = var.loc[keep_gene_mask].copy()

    n_counts = compute_n_counts(X)
    obs["n_counts"] = n_counts

    # remove cells with zero counts after geneformer vocab intersection
    keep_cell_mask = n_counts > 0
    n_cells_before = X.shape[0]
    n_cells_keep = int(keep_cell_mask.sum())

    print(f"[Prepare] Cells before filtering: {n_cells_before}")
    print(f"[Prepare] Cells with nonzero counts after filtering: {n_cells_keep}")

    if sp.issparse(X):
        X = X[keep_cell_mask].tocsr()
    else:
        X = X[keep_cell_mask]

    obs = obs.loc[keep_cell_mask].copy()

    # optional filter_pass for tokenizer
    obs["filter_pass"] = 1

    gf_adata = ad.AnnData(X=X, obs=obs, var=var)
    out_h5ad.parent.mkdir(parents=True, exist_ok=True)
    gf_adata.write_h5ad(out_h5ad)

    print(f"[Prepare] Saved Geneformer-ready h5ad to: {out_h5ad}")
    print(f"[Prepare] Final shape: {gf_adata.shape}")

    summary = {
        "input_h5ad": str(in_h5ad),
        "prepared_h5ad": str(out_h5ad),
        "ensembl_col": resolved_ensembl_col,
        "counts_source": counts_source,
        "n_genes_before": int(n_before),
        "n_genes_matched": int(n_keep),
        "n_cells_before": int(n_cells_before),
        "n_cells_keep": int(n_cells_keep),
    }

    del adata, gf_adata, X, obs, var
    gc.collect()
    return summary


def tokenize_geneformer_h5ad(
    prepared_h5ad: Path,
    tokenized_out_dir: Path,
    dataset_name: str,
    batch_key: Optional[str],
    label_key: Optional[str],
):
    """
    Tokenize one Geneformer-ready h5ad into a HuggingFace .dataset
    """
    tmp_input_dir = tokenized_out_dir / f"{dataset_name}_tmp_input"
    tmp_input_dir.mkdir(parents=True, exist_ok=True)

    tmp_h5ad = tmp_input_dir / f"{dataset_name}.h5ad"
    if prepared_h5ad != tmp_h5ad:
        if tmp_h5ad.exists() or tmp_h5ad.is_symlink():
            tmp_h5ad.unlink()
        os.symlink(prepared_h5ad, tmp_h5ad)

    # Build custom_attr exactly like the provided script, but only using columns
    # available in the prepared pseudo-control h5ad.
    prepared = sc.read_h5ad(prepared_h5ad, backed="r")
    custom_attr = existing_obs_attrs(prepared, batch_key=batch_key, label_key=label_key)
    prepared.file.close()

    print(f"\n[Tokenize] Dataset: {dataset_name}")
    print(f"[Tokenize] custom_attr_name_dict = {custom_attr}")

    tk = TranscriptomeTokenizer(
        custom_attr_name_dict=custom_attr,
        nproc=TOKENIZE_NPROC,
        chunk_size=TOKENIZE_CHUNK_SIZE,
        token_dictionary_file=TOKEN_DICT_PATH,
    )

    tokenized_out_dir.mkdir(parents=True, exist_ok=True)

    tk.tokenize_data(
        data_directory=tmp_input_dir,
        output_directory=tokenized_out_dir,
        output_prefix=dataset_name,
        file_format="h5ad",
        use_generator=USE_GENERATOR,
    )

    tokenized_path = (tokenized_out_dir / dataset_name).with_suffix(".dataset")
    print(f"[Tokenize] Saved tokenized dataset to: {tokenized_path}")
    return tokenized_path

[Info] Loaded Geneformer token dictionary with 25424 valid genes.


In [5]:
# ------------------------------------------------------------
# EMBEDDING EXTRACTION FROM PROVIDED SCRIPT
# ------------------------------------------------------------

def load_model(model_path: str):
    model = BertForMaskedLM.from_pretrained(
        model_path,
        output_hidden_states=True,
        output_attentions=False,
    )
    model.eval()
    model.to(DEVICE)
    return model


def mean_pool_nonpadding(hidden_states, lengths):
    """
    hidden_states: [B, L, D]
    lengths: [B]
    """
    pooled = []
    for i in range(hidden_states.size(0)):
        pooled.append(hidden_states[i, : lengths[i], :].mean(dim=0))
    return torch.stack(pooled, dim=0)


def extract_cell_embeddings_from_dataset(
    model,
    tokenized_dataset_path: Path,
    out_npy: Path,
    out_obs_csv: Path,
    emb_layer: int = -2,
    forward_batch_size: int = 64,
    pad_token_id: int = 0,
):
    """
    Extract one mean-pooled cell embedding per cell from tokenized dataset.
    emb_layer=-2 corresponds to 2nd-to-last hidden layer.
    """
    print(f"\n[Embed] Loading tokenized dataset: {tokenized_dataset_path}")
    ds = load_from_disk(str(tokenized_dataset_path))

    # Sort for slightly better padding efficiency, exactly as in the provided script.
    ds = ds.sort("length")

    n = len(ds)
    print(f"[Embed] Number of cells: {n}")
    print(f"[Embed] Columns: {ds.column_names}")

    model_input_size = pu.get_model_input_size(model)

    all_embs = []
    obs_cols = [c for c in ds.column_names if c not in ["input_ids", "length"]]
    obs_df = pd.DataFrame({c: ds[c] for c in obs_cols}) if len(obs_cols) > 0 else pd.DataFrame(index=np.arange(n))

    out_npy.parent.mkdir(parents=True, exist_ok=True)
    out_obs_csv.parent.mkdir(parents=True, exist_ok=True)

    for start in tqdm(range(0, n, forward_batch_size), desc="Embedding batches"):
        end = min(start + forward_batch_size, n)
        minibatch = ds.select(range(start, end))

        lengths = torch.tensor(minibatch["length"], device=DEVICE)
        max_len = int(max(minibatch["length"]))

        minibatch.set_format(type="torch")
        input_ids = minibatch["input_ids"]

        padded = pu.pad_tensor_list(
            input_ids,
            max_len,
            pad_token_id,
            model_input_size,
        )

        attention_mask = pu.gen_attention_mask(minibatch)

        with torch.no_grad():
            outputs = model(
                input_ids=padded.to(DEVICE),
                attention_mask=attention_mask,
            )

        hidden = outputs.hidden_states[emb_layer]   # [B, L, D]
        pooled = mean_pool_nonpadding(hidden, lengths).cpu().numpy()
        all_embs.append(pooled)

        del minibatch, input_ids, padded, attention_mask, outputs, hidden, pooled, lengths
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    embs = np.concatenate(all_embs, axis=0)

    # The provided script sorts the tokenized dataset by length for efficiency.
    # For pseudo-control workflows, we restore embedding rows to the prepared h5ad order
    # when adata_order is available. This keeps downstream pairing and UMAP alignment correct.
    if "adata_order" in obs_df.columns:
        obs_df["_sorted_embedding_row"] = np.arange(obs_df.shape[0], dtype=np.int64)
        order = np.argsort(obs_df["adata_order"].astype(int).values)
        embs = embs[order]
        obs_df = obs_df.iloc[order].reset_index(drop=True)

    np.save(out_npy, embs)
    obs_df.to_csv(out_obs_csv, index=False)

    print(f"[Embed] Saved embeddings: {out_npy}")
    print(f"[Embed] Saved metadata:   {out_obs_csv}")
    print(f"[Embed] Embedding shape: {embs.shape}")

    return embs, obs_df


def write_embeddings_back_to_h5ad(
    original_h5ad: Path,
    prepared_h5ad: Path,
    emb_npy: Path,
    out_h5ad: Path,
    obsm_key: str = "X_geneformer",
):
    """
    Write embeddings back to the filtered/prepared cell set order.
    Since preparation may drop genes and zero-count cells, this uses the prepared file.
    """
    adata = sc.read_h5ad(prepared_h5ad)
    embs = np.load(emb_npy)

    if adata.n_obs != embs.shape[0]:
        raise ValueError(
            f"Cell number mismatch: prepared_h5ad has {adata.n_obs} cells, "
            f"but embeddings have shape {embs.shape}"
        )

    adata.obsm[obsm_key] = embs
    out_h5ad.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(out_h5ad)
    print(f"[Write] Saved h5ad with Geneformer embeddings: {out_h5ad}")
    del adata
    gc.collect()

In [6]:
# ------------------------------------------------------------
# RUN ONE PSEUDO-CONTROL VARIANT
# ------------------------------------------------------------

def run_one_dataset(name: str, cfg: Dict):
    print("=" * 80)
    print(f"Running pseudo-control variant: {name}")
    print("=" * 80)

    prepared_h5ad = TOKENIZED_DIR / f"{name}_geneformer_ready.h5ad"
    tokenized_dataset = (TOKENIZED_DIR / name).with_suffix(".dataset")

    # Variant-specific output folder to avoid collisions and non-existent-directory errors.
    emb_variant_dir = EMB_DIR / name
    emb_npy = emb_variant_dir / f"{name}_geneformer.npy"
    obs_csv = emb_variant_dir / f"{name}_obs.csv"
    out_h5ad = UPDATED_H5AD_DIR / f"{name}_with_geneformer.h5ad"
    metadata_json = emb_variant_dir / f"{name}_geneformer_metadata.json"

    emb_variant_dir.mkdir(parents=True, exist_ok=True)

    prep_summary = None
    if prepared_h5ad.exists() and not OVERWRITE_PREPARED:
        print(f"[Prepare] Reusing existing prepared h5ad: {prepared_h5ad}")
    else:
        prep_summary = prepare_geneformer_input(
            in_h5ad=cfg["h5ad"],
            out_h5ad=prepared_h5ad,
            ensembl_col=cfg["ensembl_col"],
            counts_source=cfg["counts_source"],
            batch_key=cfg["batch_key"],
            label_key=cfg["label_key"],
        )

    if tokenized_dataset.exists() and not OVERWRITE_TOKENIZED:
        print(f"[Tokenize] Reusing existing tokenized dataset: {tokenized_dataset}")
    else:
        tokenize_geneformer_h5ad(
            prepared_h5ad=prepared_h5ad,
            tokenized_out_dir=TOKENIZED_DIR,
            dataset_name=name,
            batch_key=cfg["batch_key"],
            label_key=cfg["label_key"],
        )

    if emb_npy.exists() and obs_csv.exists() and not OVERWRITE_EMBEDDINGS:
        print(f"[Embed] Reusing existing embeddings: {emb_npy}")
        embs = np.load(emb_npy, mmap_mode="r")
        emb_shape = tuple(embs.shape)
        del embs
    else:
        model = load_model(MODEL_PATH)

        embs, obs_df = extract_cell_embeddings_from_dataset(
            model=model,
            tokenized_dataset_path=tokenized_dataset,
            out_npy=emb_npy,
            out_obs_csv=obs_csv,
            emb_layer=EMB_LAYER,
            forward_batch_size=FORWARD_BATCH_SIZE,
            pad_token_id=PAD_TOKEN_ID,
        )
        emb_shape = tuple(embs.shape)

        del model, embs, obs_df
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if WRITE_H5AD_WITH_EMB:
        if out_h5ad.exists() and not OVERWRITE_H5AD_WITH_EMB:
            print(f"[Write] Reusing existing h5ad with embeddings: {out_h5ad}")
        else:
            write_embeddings_back_to_h5ad(
                original_h5ad=cfg["h5ad"],
                prepared_h5ad=prepared_h5ad,
                emb_npy=emb_npy,
                out_h5ad=out_h5ad,
                obsm_key=OBSM_KEY,
            )

    metadata = {
        "variant_slug": name,
        "variant": cfg["variant"],
        "input_h5ad": str(cfg["h5ad"]),
        "prepared_h5ad": str(prepared_h5ad),
        "tokenized_dataset": str(tokenized_dataset),
        "embedding_npy": str(emb_npy),
        "obs_csv": str(obs_csv),
        "h5ad_with_embedding": str(out_h5ad) if WRITE_H5AD_WITH_EMB else None,
        "obsm_key": OBSM_KEY,
        "embedding_shape": list(emb_shape),
        "model_path": MODEL_PATH,
        "token_dict_path": TOKEN_DICT_PATH,
        "emb_layer": EMB_LAYER,
        "forward_batch_size": FORWARD_BATCH_SIZE,
        "prep_summary": prep_summary,
    }

    with open(metadata_json, "w") as f:
        import json
        json.dump(metadata, f, indent=2)

    return metadata


def build_pseudo_dataset_configs(pseudo_df: pd.DataFrame) -> Dict[str, Dict]:
    datasets = {}

    for _, row in pseudo_df.iterrows():
        name = row["variant_slug"]
        h5ad_path = Path(row["h5ad"])

        datasets[name] = {
            "variant": row["variant"],
            "h5ad": h5ad_path,
            "ensembl_col": ENSEMBL_COL,
            "counts_source": COUNTS_SOURCE,
            "batch_key": BATCH_KEY,
            "label_key": LABEL_KEY,
        }

    return datasets


DATASETS = build_pseudo_dataset_configs(pseudo_df)
print(f"[Config] Built DATASETS with {len(DATASETS)} variant(s).")
for k, v in DATASETS.items():
    print(k, "->", v["h5ad"])


def main():
    results = []

    for name, cfg in DATASETS.items():
        try:
            result = run_one_dataset(name, cfg)
            result["status"] = "done"
            result["error"] = ""
            results.append(result)
        except Exception as e:
            print(f"[Error] Failed variant {name}: {e}")
            results.append({
                "variant_slug": name,
                "variant": cfg.get("variant", ""),
                "input_h5ad": str(cfg.get("h5ad", "")),
                "status": "failed",
                "error": repr(e),
            })
            if not CONTINUE_ON_ERROR:
                raise

    manifest = pd.DataFrame(results)
    manifest_path = MANIFEST_DIR / "geneformer_strict_embedding_manifest.csv"
    manifest.to_csv(manifest_path, index=False)
    print(f"[Manifest] Saved: {manifest_path}")
    display(manifest)
    return manifest


manifest = main()

[Config] Built DATASETS with 1 variant(s).
S5_SEACell_OT_sampled_average__nmc_500__topk_05__seed_000 -> /ibex/project/c2366/Perturb_data/Replogle_k562_data/Replogle_K562_essential/single/S5_SEACell_OT_sampled_average/nmc_500/topk_05/seed_000/pseudo_control_aligned_to_perturbed.h5ad
Running pseudo-control variant: S5_SEACell_OT_sampled_average__nmc_500__topk_05__seed_000

[Prepare] Reading: /ibex/project/c2366/Perturb_data/Replogle_k562_data/Replogle_K562_essential/single/S5_SEACell_OT_sampled_average/nmc_500/topk_05/seed_000/pseudo_control_aligned_to_perturbed.h5ad
[Prepare][Warn] Label key 'perturbation_label' not found in adata.obs. It will not be tokenized.
[Prepare] Ensembl source column: ensembl_id
[Prepare] Genes before filtering: 8563
[Prepare] Genes matched Geneformer vocab: 8221
[Prepare] Cells before filtering: 299694
[Prepare] Cells with nonzero counts after filtering: 299694
[Prepare] Saved Geneformer-ready h5ad to: /ibex/project/c2366/Perturb_data/Replogle_k562_data/Replog

,variant_slug,variant,input_h5ad,status,error
0,S5_SEACell_OT_sampled_average__nmc_500__topk_0...,S5_SEACell_OT_sampled_average/nmc_500/topk_05/...,/ibex/project/c2366/Perturb_data/Replogle_k562...,failed,ArrowInvalid('Value 2147489393 too large to fi...


In [7]:
import datasets

data = datasets.load_from_disk("/ibex/project/c2366/Perturb_data/Replogle_k562_data/Replogle_K562_essential/single/_geneformer_strict_script_embeddings/tokenized/S5_SEACell_OT_sampled_average__nmc_350__topk_05__seed_000.dataset")

FileNotFoundError: Directory /ibex/project/c2366/Perturb_data/Replogle_k562_data/Replogle_K562_essential/single/_geneformer_strict_script_embeddings/tokenized/S5_SEACell_OT_sampled_average__nmc_350__topk_05__seed_000.dataset not found

In [ ]:
data

Dataset({
    features: ['input_ids', 'condition', 'perturbation', 'nperts', 'gene', 'guide_id', 'source_obs_name', 'adata_order', 'length'],
    num_rows: 1024
})